In [1]:
import os
import torch
from pathlib import Path

from torch.mtia import snapshot
from torch.xpu import device
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, TrainerCallback
from datasets import load_dataset
from perf_estimator.trainer.plugins import ProfilerCallback, SnapshotCallback
from ures.string import format_memory

/home/glaswegian/miniconda3/envs/xmem-12.8/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# os.environ["TORCH_USE_CUDA_DSA"] = "1"


In [3]:
# Configure logging
# ---------------------------
# 1. Model Initialization (from scratch)
# ---------------------------
# We use the configuration of a popular small-scale LLM (facebook/opt-125m)
# but initialize the model randomly (i.e. train from scratch)
model_name = "facebook/opt-125m"
# model_name = "EleutherAI/gpt-neo-125M"
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
config = AutoConfig.from_pretrained(model_name)  # load config; do NOT load pretrained weights
model = AutoModelForCausalLM.from_config(config)   # randomly initialized model
print(model)

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 768, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 768)
      (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-11): 12 x OPTDecoderLayer(
          (self_attn): OPTSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_norm): LayerNorm((768,)

In [4]:
model_size = 0
parameters_list = list(model.parameters())
parameters_list.reverse()
for tensor in parameters_list:
    para_size = tensor.nelement() * tensor.element_size()
    model_size += para_size
    print(f"{tensor.shape}: size: {format_memory(para_size)}")
print(f"Model size: {format_memory(model_size)}")

torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768, 3072]): size: 9.00 MB
torch.Size([3072]): size: 12.00 KB
torch.Size([3072, 768]): size: 9.00 MB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768]): size: 3.00 KB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768]): size: 3.00 KB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768]): size: 3.00 KB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768, 3072]): size: 9.00 MB
torch.Size([3072]): size: 12.00 KB
torch.Size([3072, 768]): size: 9.00 MB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768]): size: 3.00 KB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768]): size: 3.00 KB
torch.Size([768, 768]): size: 2.25 MB
torch.Size([768]):

In [10]:
# Load tokenizer (we can reuse the pretrained tokenizer)
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token  # assign PAD token if missing

# Enable gradient checkpointing to reduce memory usage (at the cost of additional compute)
model.gradient_checkpointing_enable()

# ---------------------------
# 2. Dataset Preparation
# ---------------------------
# Load the Wikitext-2 dataset as our general-purpose text corpus.
# For a quick profiling run, we use only a small subset.
dataset = load_dataset(
    "wikitext", "wikitext-2-raw-v1",
)
train_dataset = dataset["train"].select(range(1000))  # limit to 1000 examples for this demo

# Tokenization: convert text to token IDs (truncated to a maximum length)
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, max_length=128)

tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Data collator: handles padding and prepares labels for causal LM (labels equal to input_ids)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ---------------------------
# 3. Trainer Setup with Memory Profiling Callback
# ---------------------------
# TrainingArguments are set to run only 3 steps and use a small batch size suitable for an 8–12GB GPU.
training_args = TrainingArguments(
    output_dir="output",
    # per_device_train_batch_size=2,
    max_steps=3,  # run only 3 training iterations for profiling
    # gradient_accumulation_steps=1,
    fp16=False,  # using full precision; set True if your GPU supports mixed precision to save memory
    # report_to=[],  # disable external logging (e.g., wandb)
    disable_tqdm=False,
    use_cpu=False,
    do_train=True,
    do_eval=False,
    save_strategy="no",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    callbacks=[ProfilerCallback(), SnapshotCallback()],
)

# ---------------------------
# 4. Training with PyTorch Profiler
# ---------------------------

trainer.train()  # run 3 training iterations

# Print a summary of the profiler's memory usage by CUDA operation (top 10 ops)
print("Profiler Memory Usage Summary (top CUDA ops):")


Starting profiler...


Step,Training Loss


Stopping profiler...
Profiler Memory Usage Summary (top CUDA ops):


In [6]:
from ures.files import filter_files
p_file = filter_files("pt.trace.json", str(Path().home().joinpath("DL-Estimator")), fuzz=True)[-1]
p_file


'/home/glaswegian/DL-Estimator/20250406-212639-f41d/results/callback/Profiler/glaswegian-Z890-GAMING-X-WIFI7_15684.1743974807975840398.pt.trace.json'

In [7]:
from perf_estimator.estimator import TrainerEstimator, Estimator
from perf_estimator.dataset import image_dataset
from perf_estimator.config import Config
_config = Config()
_config.trainer.huggingface_enable = False
_config.trainer.huggingface_model_name = model_name
estimator = TrainerEstimator(
    dataloader=image_dataset(batch=100),
    profiler_file=p_file,
    max_gpu_memory_in_gb=8,
    config=_config,
)
estimator

Duplicate layer name found: OPTDecoderLayer_11. Renaming to OPTDecoderLayer_11_f3
Duplicate layer name found: LayerNorm_22. Renaming to LayerNorm_22_09
Duplicate layer name found: OPTSdpaAttention_11. Renaming to OPTSdpaAttention_11_67
Duplicate layer name found: Linear_66. Renaming to Linear_66_13
Duplicate layer name found: Linear_67. Renaming to Linear_67_78
Duplicate layer name found: Linear_68. Renaming to Linear_68_f0
Duplicate layer name found: Linear_69. Renaming to Linear_69_d8
Duplicate layer name found: LayerNorm_23. Renaming to LayerNorm_23_36
Duplicate layer name found: Linear_70. Renaming to Linear_70_0e
Duplicate layer name found: ReLU_11. Renaming to ReLU_11_74
Duplicate layer name found: Linear_71. Renaming to Linear_71_5d
Duplicate layer name found: OPTDecoderLayer_10. Renaming to OPTDecoderLayer_10_c8
Duplicate layer name found: LayerNorm_20. Renaming to LayerNorm_20_2c
Duplicate layer name found: OPTSdpaAttention_10. Renaming to OPTSdpaAttention_10_f8
Duplicate laye

In [8]:

allocator, est_result = estimator.estimate()


ValueError: Time 453209388536.003, the amount of memory freed is not the same as the current block.Original Size: 512 != free Size: 512

In [10]:
allocator.plot_memory_change()

In [11]:
layer = estimator.profiler.get_iteration(1).layer_summary()